In [1]:
from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser, PydanticOutputParser
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, BaseMessage
from langgraph.checkpoint.memory import MemorySaver

In [2]:
#Specialised reducer
from langgraph.graph.message import add_messages

In [3]:
from typing import Literal, TypedDict, Optional, Annotated

class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

In [4]:
llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite", temperature=0.7, max_output_tokens=512)

In [5]:
def chat_with_ai(state: ChatState):
    messages = state['messages']
    prompt_template = PromptTemplate(
        input_variables=["messages"],
        template="You are a helpful assistant. Continue the conversation based on the following messages. Reply in short-oneline if possi:\n\n{messages}"
    )
    chain = prompt_template | llm | StrOutputParser()
    response = chain.invoke({"messages" : messages})
    return  {"messages": [AIMessage(content=response)]}
    

In [6]:
graph = StateGraph(ChatState)
checkpointer = MemorySaver()

In [7]:
graph.add_node('chat_node' , chat_with_ai)

#add_edges
graph.add_edge(START, 'chat_node')
graph.add_edge('chat_node', END)

In [8]:
workflow = graph.compile(checkpointer=checkpointer)

In [9]:
input_state = {
    "messages" : [
        SystemMessage(content="You are a Pre-historic animals expert. Be precise and concise in your answers."),
        HumanMessage(content="What is the largest dinosaur that ever lived?")
    ]
}

In [10]:
final_state = workflow.invoke(input_state)

ValueError: Checkpointer requires one or more of the following 'configurable' keys: thread_id, checkpoint_ns, checkpoint_id

In [11]:
for message in final_state['messages']:
    print(f"{message.type}: {message.content}")

NameError: name 'final_state' is not defined

In [12]:
chat_history = [SystemMessage(content="You are a helpful assistant. Be precise and concise in your answers. Whenever possible, reply in short one-line answers.")]

In [13]:
thread_id = "user_1"

In [14]:
while True:
    input_text = input()
    print(f"User: {input_text}")
    list_of_words = input_text.strip().lower().split(" ")
    end_words = ["exit", "quit", "bye"]
    if any(word in list_of_words for word in end_words):
        print("Exiting the chat. Goodbye!")
        break

    config = {"configurable" : {"thread_id" : thread_id}}
    final_state = workflow.invoke({"messages": [HumanMessage(content=input_text)] }, config=config)

    print(f"AI: {final_state['messages'][-1].content}")


Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


User: Hi i m Harshit
AI: Hi Harshit, how can I help you today?
User: can u guess my name
AI: Your name is Harshit!
User: bye
Exiting the chat. Goodbye!


In [15]:
workflow.get_state(config= config)

StateSnapshot(values={'messages': [HumanMessage(content='Hi i m Harshit', additional_kwargs={}, response_metadata={}, id='991e94f9-fcfa-42dd-af99-4bd2b9810133'), AIMessage(content='Hi Harshit, how can I help you today?', additional_kwargs={}, response_metadata={}, id='676579f5-0983-4253-8257-728a4bed8b37', tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='can u guess my name', additional_kwargs={}, response_metadata={}, id='4ca811e8-bc40-4698-a383-2b4211a031dd'), AIMessage(content='Your name is Harshit!', additional_kwargs={}, response_metadata={}, id='9a93eca7-0e85-4f96-8584-defe69c36359', tool_calls=[], invalid_tool_calls=[])]}, next=(), config={'configurable': {'thread_id': 'user_1', 'checkpoint_ns': '', 'checkpoint_id': '1f1aec8d-673a-6b6b-8004-a0d7f42fb173'}}, metadata={'source': 'loop', 'step': 4, 'parents': {}}, created_at='2026-09-12T16:41:43.979300+00:00', parent_config={'configurable': {'thread_id': 'user_1', 'checkpoint_ns': '', 'checkpoint_id': '1f1aec8d-4145-6c8

In [16]:
list(workflow.get_state_history(config= config))

[StateSnapshot(values={'messages': [HumanMessage(content='Hi i m Harshit', additional_kwargs={}, response_metadata={}, id='991e94f9-fcfa-42dd-af99-4bd2b9810133'), AIMessage(content='Hi Harshit, how can I help you today?', additional_kwargs={}, response_metadata={}, id='676579f5-0983-4253-8257-728a4bed8b37', tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='can u guess my name', additional_kwargs={}, response_metadata={}, id='4ca811e8-bc40-4698-a383-2b4211a031dd'), AIMessage(content='Your name is Harshit!', additional_kwargs={}, response_metadata={}, id='9a93eca7-0e85-4f96-8584-defe69c36359', tool_calls=[], invalid_tool_calls=[])]}, next=(), config={'configurable': {'thread_id': 'user_1', 'checkpoint_ns': '', 'checkpoint_id': '1f1aec8d-673a-6b6b-8004-a0d7f42fb173'}}, metadata={'source': 'loop', 'step': 4, 'parents': {}}, created_at='2026-09-12T16:41:43.979300+00:00', parent_config={'configurable': {'thread_id': 'user_1', 'checkpoint_ns': '', 'checkpoint_id': '1f1aec8d-4145-6c

In [17]:
workflow.get_state({"configurable" : {"thread_id" : "user_1" , "checkpoint_id" : "1f1aec8c-ec5f-6326-8000-30640554e74e"}})

StateSnapshot(values={'messages': [HumanMessage(content='Hi i m Harshit', additional_kwargs={}, response_metadata={}, id='991e94f9-fcfa-42dd-af99-4bd2b9810133')]}, next=('chat_node',), config={'configurable': {'thread_id': 'user_1', 'checkpoint_id': '1f1aec8c-ec5f-6326-8000-30640554e74e'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2026-09-12T16:41:31.096758+00:00', parent_config={'configurable': {'thread_id': 'user_1', 'checkpoint_ns': '', 'checkpoint_id': '1f1aec8c-ec5f-6325-bfff-990bffa6d9a7'}}, tasks=(PregelTask(id='2e5f361a-e2aa-bb81-bf7a-a3dee7ee8d3f', name='chat_node', path=('__pregel_pull', 'chat_node'), error=None, interrupts=(), state=None, result={'messages': [AIMessage(content='Hi Harshit, how can I help you today?', additional_kwargs={}, response_metadata={}, id='676579f5-0983-4253-8257-728a4bed8b37', tool_calls=[], invalid_tool_calls=[])]}),), interrupts=())

In [18]:
final_state2 = workflow.invoke(None, config={"configurable" : {"thread_id" : "user_1" , "checkpoint_id" : "1f1aec8c-ec5f-6326-8000-30640554e74e"}})

In [19]:
final_state2

{'messages': [HumanMessage(content='Hi i m Harshit', additional_kwargs={}, response_metadata={}, id='991e94f9-fcfa-42dd-af99-4bd2b9810133'),
  AIMessage(content='Hi Harshit! How can I help you today?', additional_kwargs={}, response_metadata={}, id='36761527-c156-489a-acff-cfae10ad1eeb', tool_calls=[], invalid_tool_calls=[])]}

In [20]:
list(workflow.get_state_history(config= config))

[StateSnapshot(values={'messages': [HumanMessage(content='Hi i m Harshit', additional_kwargs={}, response_metadata={}, id='991e94f9-fcfa-42dd-af99-4bd2b9810133'), AIMessage(content='Hi Harshit! How can I help you today?', additional_kwargs={}, response_metadata={}, id='36761527-c156-489a-acff-cfae10ad1eeb', tool_calls=[], invalid_tool_calls=[])]}, next=(), config={'configurable': {'thread_id': 'user_1', 'checkpoint_ns': '', 'checkpoint_id': '1f1aec95-8d3c-6793-8002-1ac85e10817e'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-09-12T16:45:22.712974+00:00', parent_config={'configurable': {'thread_id': 'user_1', 'checkpoint_ns': '', 'checkpoint_id': '1f1aec95-7bde-6fcf-8001-e9c3d42dbd8f'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'messages': [HumanMessage(content='Hi i m Harshit', additional_kwargs={}, response_metadata={}, id='991e94f9-fcfa-42dd-af99-4bd2b9810133')]}, next=('chat_node',), config={'configurable': {'thread_id': 'user_1', 'checkpoint_ns':